In [1]:
from datasets import load_dataset
import pandas as pd
from pathlib import Path
import re


/Users/tejumadeafonja/miniconda3/envs/tabularbench/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_and_update_great(dataset_config, split="test", predictions_file=None):
    """Load a Hugging Face split and add that file's pred fields to `great`."""
    if predictions_file is None:
        predictions_file = Path("../fewshot10") / f"{dataset_config}.jsonl"
    else:
        predictions_file = Path(predictions_file)

    predictions = pd.read_json(predictions_file, lines=True)

    required = {"doc_id", "pred"}
    missing = required - set(predictions.columns)
    if missing:
        raise ValueError(f"{predictions_file} is missing columns: {sorted(missing)}")
    if predictions["doc_id"].isna().any() or predictions["doc_id"].duplicated().any():
        raise ValueError("Prediction doc_id values must be present and unique.")

    ids = pd.to_numeric(predictions["doc_id"], errors="raise")
    if (ids % 1 != 0).any():
        raise ValueError("Prediction doc_id values must be integers.")
    predictions["doc_id"] = ids.astype(int)

    data = load_dataset(
        "Kanubalad/beyond_accept_or_deny",
        dataset_config,
        split=split,
    ).to_pandas().reset_index(drop=True)

    if data.empty or "great" not in data.columns:
        raise ValueError(f"{dataset_config!r} has no rows or no 'great' column.")

    expected_ids = set(range(len(data)))
    actual_ids = set(predictions["doc_id"])
    if actual_ids != expected_ids:
        raise ValueError(
            f"Prediction IDs don't match the {len(data)} {split!r} rows. "
            f"Missing IDs: {sorted(expected_ids - actual_ids)[:10]}; "
            f"extra IDs: {sorted(actual_ids - expected_ids)[:10]}"
        )

    # Reorder predictions by doc_id so each prediction goes to its matching row.
    by_id = predictions.set_index("doc_id")["pred"]
    ordered_preds = [by_id.loc[i] for i in range(len(data))]

    pred_fields = list(ordered_preds[0])
    for doc_id, pred in enumerate(ordered_preds):
        if not isinstance(pred, dict) or not pred:
            raise ValueError(f"Prediction for doc_id={doc_id} must be a non-empty object.")
        if list(pred) != pred_fields:
            raise ValueError("All prediction rows must contain the same fields.")

    # Preserve the feature fields already represented in the original `great` text.
    original_great = str(data.loc[0, "great"])
    great_fields = [
        column for column in data.columns
        if re.search(rf"(?:^|, ){re.escape(column)} is ", original_great)
    ]
    if not great_fields:
        raise ValueError("Could not identify feature fields in the existing 'great' value.")

    data["doc_id"] = range(len(data))
    for field in pred_fields:
        data[field] = [pred[field] for pred in ordered_preds]

    # Add predicted fields to `great` without duplicating existing fields.
    great_fields.extend(field for field in pred_fields if field not in great_fields)
    data["great"] = data.apply(
        lambda row: ", ".join(f"{field} is {row[field]}" for field in great_fields),
        axis=1,
    )

    # Verify each row's prediction and generated text against its doc_id.
    for doc_id, pred in enumerate(ordered_preds):
        row = data.loc[doc_id]
        for field, expected in pred.items():
            if row[field] != expected:
                raise AssertionError(f"{field} was assigned to the wrong doc_id={doc_id}.")
            if f"{field} is {expected}" not in row["great"]:
                raise AssertionError(f"{field} is missing from great for doc_id={doc_id}.")

    # Keep doc_id, the fields used in the original great, prediction fields, and great.
    output_columns = ["doc_id", *great_fields, *pred_fields, "great"]
    output_columns = list(dict.fromkeys(output_columns))
    return data.loc[:, output_columns]

In [3]:
df_german_counterfactual = load_and_update_great("german_counterfactual")
df_german_counterfactual[
    ["doc_id", "loan_amount", "loan_duration", "great"]
].head()

,doc_id,loan_amount,loan_duration,great
0,0,1197.7571,23.981827,"gender is female, age is 33.0, checking_status..."
1,1,12345.6789,60.000001,"gender is male, age is 33.0, checking_status i..."
2,2,1045.7545,12.000118,"gender is male, age is 30.0, checking_status i..."
3,3,1045.7840,24.630234,"gender is male, age is 23.0, checking_status i..."
4,4,1048.8750,18.045685,"gender is male, age is 24.0, checking_status i..."


In [4]:
df_german_counterfactual['great'][0]

'gender is female, age is 33.0, checking_status is a checking account with less than 200 Deutsche Mark, credit_history is has delayed paying back credits received from this bank in the past, purpose is radio/tv, savings_status is a savings acccount with less than 100 Deutsche Mark, employment is employed for more than one year but less than four years, other_parties is no guarantor, residence_since is 2.0000634, property_magnitude is car, other_payment_plans is none, housing is self-owned house, existing_credits is 0.99991703, job is skilled, num_dependents is 0.9999933, own_telephone is none, foreign_worker is yes, installment_commitment is 3.9986734, loan_amount is 1197.7571, loan_duration is 23.981827'

In [5]:
df_german_counterfactual.columns

Index(['doc_id', 'gender', 'age', 'checking_status', 'credit_history',
       'purpose', 'savings_status', 'employment', 'other_parties',
       'residence_since', 'property_magnitude', 'other_payment_plans',
       'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone',
       'foreign_worker', 'installment_commitment', 'loan_amount',
       'loan_duration', 'great'],
      dtype='object')

In [6]:
df_hmda_ny_pathspecific = load_and_update_great("hmda_ny_pathspecific")
df_hmda_ny_pathspecific[["doc_id", "loan_amount", "great"]].head()

,doc_id,loan_amount,great
0,0,159.0,"applicant_sex is male, applicant_race_1 is Whi..."
1,1,1000.0,"applicant_sex is male, applicant_race_1 is Whi..."
2,2,130.0,"applicant_sex is female, applicant_race_1 is W..."
3,3,120.0,"applicant_sex is male, applicant_race_1 is non..."
4,4,1000.0,"applicant_sex is male, applicant_race_1 is Whi..."


In [7]:
df_hmda_ny_pathspecific['great'][0]

'applicant_sex is male, applicant_race_1 is White, loan_type is 0, property_type is 0, loan_purpose is 2, owner_occupancy is 0, msamd is 45060, census_tract_number is 206.0, applicant_income_000s is 78, purchaser_type is 0, hud_median_family_income is 68000, tract_to_msamd_income is 89.76, number_of_owner_occupied_units is 1587, number_of_1_to_4_family_units is 2466, preapproval is 2, loan_amount is 159.0'

In [8]:
df_hmda_ny_pathspecific.shape

(16665, 18)

In [9]:
df_hmda_ny_pathspecific.columns

Index(['doc_id', 'applicant_sex', 'applicant_race_1', 'loan_type',
       'property_type', 'loan_purpose', 'owner_occupancy', 'msamd',
       'census_tract_number', 'applicant_income_000s', 'purchaser_type',
       'hud_median_family_income', 'tract_to_msamd_income',
       'number_of_owner_occupied_units', 'number_of_1_to_4_family_units',
       'preapproval', 'loan_amount', 'great'],
      dtype='object')

In [10]:
from datasets import Dataset, load_dataset

REPO_ID = "tej/beyond_accept_or_deny"


def upload_updated_config(dataset_config, split="test"):
    # Build the updated DataFrame using the loader function.
    df = load_and_update_great(dataset_config, split=split)

    # Refuse to upload if the row IDs are not aligned with the HF split.
    expected_ids = list(range(len(df)))
    if df["doc_id"].tolist() != expected_ids:
        raise ValueError(f"{dataset_config}: doc_id values do not match row order.")

    new_config = f"{dataset_config}_with_predictions"

    # Publish as a separate config, leaving the original config unchanged.
    Dataset.from_pandas(df, preserve_index=False).push_to_hub(
        REPO_ID,
        config_name=new_config,
        split=split,
    )

    # Read it back from Hugging Face and check the upload.
    uploaded = load_dataset(REPO_ID, new_config, split=split)
    if len(uploaded) != len(df):
        raise AssertionError(
            f"Row count mismatch: uploaded {len(uploaded)}, expected {len(df)}."
        )

    for column in ("doc_id", "great"):
        if column not in uploaded.column_names:
            raise AssertionError(f"Uploaded dataset is missing column {column!r}.")

    if uploaded[0]["doc_id"] != df.iloc[0]["doc_id"]:
        raise AssertionError("Uploaded first row does not match the local data.")
    if uploaded[len(df) - 1]["doc_id"] != df.iloc[-1]["doc_id"]:
        raise AssertionError("Uploaded last row does not match the local data.")

    print(f"Uploaded and verified: {REPO_ID} / {new_config} / {split}")
    return uploaded

In [11]:
for dataset in ("german", "hmda_ny", "hmda_tx", "homecredit"):
    for exp in ("counterfactual", "pathspecific"):
        print(f"Uploading updated config for {dataset}_{exp}...")
        print("_" * 80)
        dataset_config = f"{dataset}_{exp}"
        _ = upload_updated_config(dataset_config)

Uploading updated config for german_counterfactual...
________________________________________________________________________________


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 271.74ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Upload 0 LFS files: 0it [00:00, ?it/s]
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  3.33it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
Generating test split: 100%|██████████| 100/100 [00:00<00:00, 4906.54 examples/s]


Uploaded and verified: tej/beyond_accept_or_deny / german_counterfactual_with_predictions / test
Uploading updated config for german_pathspecific...
________________________________________________________________________________


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 354.10ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Upload 0 LFS files: 0it [00:00, ?it/s]
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  3.42it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
Generating test split: 100%|██████████| 100/100 [00:00<00:00, 14194.40 examples/s]


Uploaded and verified: tej/beyond_accept_or_deny / german_pathspecific_with_predictions / test
Uploading updated config for hmda_ny_counterfactual...
________________________________________________________________________________


...

Uploaded and verified: tej/beyond_accept_or_deny / hmda_tx_pathspecific_with_predictions / test
Uploading updated config for homecredit_counterfactual...
________________________________________________________________________________


Creating parquet from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 421.33ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Upload 0 LFS files: 0it [00:00, ?it/s]
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  3.24it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
Generating test split: 100%|██████████| 9887/9887 [00:00<00:00, 346340.53 examples/s]


Uploaded and verified: tej/beyond_accept_or_deny / homecredit_counterfactual_with_predictions / test
Uploading updated config for homecredit_pathspecific...
________________________________________________________________________________


Creating parquet from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 413.57ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Upload 0 LFS files: 0it [00:00, ?it/s]
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  2.69it/s]
No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded and verified: tej/beyond_accept_or_deny / homecredit_pathspecific_with_predictions / test


In [12]:
data = load_dataset(
        "tej/beyond_accept_or_deny",
        "homecredit_pathspecific_with_predictions",
        split="test",
    ).to_pandas().reset_index(drop=True)

In [13]:
data

,doc_id,CODE_GENDER,age,EXT_SOURCE_2,EXT_SOURCE_3,EXT_SOURCE_1,NAME_EDUCATION_TYPE,DAYS_EMPLOYED,NAME_INCOME_TYPE,ORGANIZATION_TYPE,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,loan_amount,annuity_amount,great
0,0,M,25.901371,0.594121,0.531686,0.522744,Higher education,0.868493,State servant,Emergency,675000.0,0.011703,675000.0,23750.000,"CODE_GENDER is M, age is 25.901371, EXT_SOURCE..."
1,1,F,36.120550,0.397804,0.740799,0.494439,Secondary / secondary special,11.986301,Working,Self-employed,900000.0,0.018634,900000.0,30000.000,"CODE_GENDER is F, age is 36.12055, EXT_SOURCE_..."
2,2,F,42.747944,0.705121,0.275000,0.328901,Higher education,22.523287,Working,Police,1084500.0,0.011703,1084500.0,25852.812,"CODE_GENDER is F, age is 42.747944, EXT_SOURCE..."
3,3,M,44.961643,0.797548,0.481249,0.704410,Secondary / secondary special,2.501370,Commercial associate,Business Entity Type 3,900000.0,0.032561,900000.0,30000.000,"CODE_GENDER is M, age is 44.961643, EXT_SOURCE..."
4,4,M,48.550686,0.561858,0.350696,0.811711,Higher education,15.931507,Commercial associate,Business Entity Type 3,1800000.0,0.020713,1800000.0,60000.000,"CODE_GENDER is M, age is 48.550686, EXT_SOURCE..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9882,9882,F,47.441093,0.519623,0.586740,0.670483,Secondary / secondary special,1.216438,State servant,Other,423000.0,0.019689,422972.2,24063.110,"CODE_GENDER is F, age is 47.441093, EXT_SOURCE..."
9883,9883,M,50.342464,0.114403,0.703203,0.729517,Secondary / secondary special,7.145206,Working,Self-employed,67500.0,0.020713,675000.0,33750.000,"CODE_GENDER is M, age is 50.342464, EXT_SOURCE..."
9884,9884,F,28.487670,0.127466,0.598926,0.208848,Secondary / secondary special,4.005479,State servant,Transport: type 2,450000.0,0.019689,450000.0,18000.000,"CODE_GENDER is F, age is 28.48767, EXT_SOURCE_..."
9885,9885,M,38.726030,0.669362,0.510089,0.746980,Secondary / secondary special,18.553425,Working,Business Entity Type 3,940500.0,0.009657,940575.0,32302.500,"CODE_GENDER is M, age is 38.72603, EXT_SOURCE_..."
